In [ ]:
try:
    from google.colab import drive
    drive.mount('/gdrive')
    dataset_root = '/gdrive/MyDrive/datasets'
    !pip install torchinfo tqdm
    colab = True
except Exception as e:
    print(e)
    print('Assuming we\'re not on colab.')
    dataset_root = './datasets'
    colab = False

print('Will store datasets in', dataset_root)

import os

if os.name == 'nt':
    print("Disabling multiprocessing because we're running on windows.")
    cpu_num = 0
elif colab:
    cpu_num = 2
else:
    cpu_num = os.cpu_count() // 2
    print('Dataloaders will use {} CPUs'.format(cpu_num))

In [ ]:
import torch
import torch.utils.data as tud
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms.functional as tvf
import torchvision.datasets as tds
import torchvision.utils as tu

from tqdm import tqdm
from torchinfo import summary
import matplotlib.pyplot as plt

In [ ]:
mnist_train = tds.MNIST(
    root=dataset_root,
    download=True,
    train=True,
    transform=tvf.pil_to_tensor,
)

mnist_eval = tds.MNIST(
    root=dataset_root,
    download=True,
    train=False,
    transform=tvf.pil_to_tensor,
)

In [ ]:
batchsize = 32

train_loader = tud.DataLoader(mnist_train, batch_size=batchsize, num_workers=cpu_num, shuffle=True)
val_loader = tud.DataLoader(mnist_eval, batch_size=batchsize, shuffle=True)

In [ ]:
class MyNn(nn.Module):
    def __init__(self, in_sz, out_sz, hidden_sz=128):
        super().__init__()
        self.in_sz = in_sz
        self.out_sz = out_sz
        self.lin1 = nn.Linear(in_sz, hidden_sz, bias=True)
        self.lin2 = nn.Linear(hidden_sz, hidden_sz, bias=True)
        self.lin3 = nn.Linear(hidden_sz, hidden_sz, bias=True)
        self.lin4 = nn.Linear(hidden_sz, out_sz, bias=True)

    def forward(self, x):
        x = x.view(-1, self.in_sz)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        x = F.relu(x)
        x = self.lin3(x)
        x = F.relu(x)
        x = self.lin4(x)
        return x

In [ ]:
class MyCnn(nn.Module):
    def __init__(self, out_sz, ch=32):
        super().__init__()

        self.channels = ch
        self.out_sz = out_sz
        # (28x28)
        self.conv1_1 = nn.Conv2d(1, ch, kernel_size=3, stride=1, padding=1)
        self.conv1_2 = nn.Conv2d(ch, ch*2, kernel_size=3, stride=1, padding=1)

        # (14x14)
        self.lin1 = nn.Linear(ch*2 * 14 * 14, 128, bias=True)
        self.lin2 = nn.Linear(128, out_sz, bias=True)

    def forward(self, x):
        x = self.conv1_1(x)
        x = F.relu(x)
        x = self.conv1_2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = x.view(-1, self.channels*2 * 14 * 14)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        return x

In [ ]:
%%time
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# model = MyNn(28 * 28, 10, hidden_sz=32).to(device)
model = MyCnn(out_sz=10, ch=16).to(device)

print(summary(model, (1, 1, 28, 28)))
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
# lossfn = nn.CrossEntropyLoss()
lossfn = nn.MSELoss()
epochs = 15

loss_plot = []
for epoch in tqdm(range(epochs)):
    for i, (images, target) in enumerate(train_loader):
        optimizer.zero_grad()
        images = images.float().to(device)
        targets = target.to(device)

        outs = model(images)
        # loss = lossfn(outs, targets) # If using CrossEntropy
        loss = lossfn(F.softmax(outs, dim=1), F.one_hot(targets, 10).float()) # If using MSE
        loss.backward()
        optimizer.step()

    losses = []
    for i, (images, target) in enumerate(val_loader):
        with torch.no_grad():
            images = images.float().to(device)
            targets = target.to(device)
            outs = model(images)
            # loss = lossfn(outs, targets) # If using CrossEntropy
            loss = lossfn(F.softmax(outs, dim=1), F.one_hot(targets, 10).float()) # If using MSE
            losses.append(loss)
    epoch_loss = torch.Tensor(losses).mean().item()
    print(epoch_loss)
    loss_plot.append(epoch_loss)

plt.plot(loss_plot)

In [ ]:
total = len(mnist_eval)
correct = 0
with torch.no_grad():
    for image, target in mnist_eval:
        pred = model(image.float().to(device))
        pred = F.softmax(pred, dim=1).argmax()
        if pred == target:
            correct += 1
        
    print("{:.2f}% correct".format(100*correct/total))

In [ ]:
# !pip install scikit-learn 
from sklearn import metrics

evals = mnist_eval
right = 0
total = 0
y_pred=[]
y_true=[]

model.eval()
with torch.no_grad():
    for image, target in evals:
        pred = model(image.float().to(device))
        pred = F.softmax(pred, dim=1).argmax()
        
        y_pred.append(pred.item())
        y_true.append(target)

metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, normalize='true')

In [ ]:
from ipywidgets import interact

classes = {
    0:  "T-shirt/top",
    1: 	"Trouser",
    2: 	"Pullover",
    3: 	"Dress",
    4: 	"Coat",
    5: 	"Sandal",
    6: 	"Shirt",
    7: 	"Sneaker",
    8: 	"Bag",
    9: 	"Ankle boot",
}

@interact(index=(0, len(mnist_eval) - 1, 1))
def draw_preds(index=0):
    with torch.no_grad():
        image = mnist_eval[index][0]
        pred = model(image.float().to(device))
        pred = F.softmax(pred, dim=1)
        clsid = pred.argmax()
        plt.imshow(image.float().cpu().squeeze(), cmap='gray')
        print(classes[int(clsid)])

# As homework...
Please scale the CNN model to your liking. Along with increasing the number of layers and the number of channels, you may change any other hyperparameters. Shoot for at least 90% accuracy

Be prepared to answer the following questions
* What changes did you make and why?
* How do your results change? Refer to the confusion matrix.
* Did your changes improve the classification accuracy of the worst labels?
* Did your model overfit? How did you know?

Next, go back and use the linear model (with only nn.linear blocks) and scale it to reach the same evaluation accuracy as your convnet, changing other parameters as needed. Using the model summary cell, compare the number of trainable parameters and total mult & adds.